# Avaliação de IoU — Versão 1 (antiga, histórica)


> ⚠️ **Nota:** este notebook contém a primeira versão do código de avaliação de IoU,
> que calculava apenas um IoU médio por imagem, sem separar bounding box de máscara
> e sem contabilizar TP/FP/FN. Foi substituído pela versão em `avaliacao_iou_v2_atual.ipynb`,
> que usa pareamento guloso por instância com limiar de IoU. Mantido aqui apenas para
> referência histórica — não deve ser executado para gerar resultados válidos (contém,
> inclusive, um caminho de dataset hard-coded de um experimento anterior que não reflete
> o dataset atual).


In [ ]:
import os
import cv2
import numpy as np


def label_to_mask(label_path, img_shape, class_filter=None):
    """
    Constrói uma máscara binária a partir de um rótulo YOLOv8-seg (Roboflow).

    label_path : caminho do arquivo .txt de rótulo (uma linha por instância:
                 class_id x1 y1 x2 y2 ... xn yn, coordenadas normalizadas 0-1)
    img_shape  : (altura, largura) da imagem correspondente
    class_filter : int, lista de ints, ou None.
                   Se None, todas as classes são combinadas numa única máscara
                   binária (foreground x fundo). Se informado, só as instâncias
                   dessa(s) classe(s) entram na máscara.
    """
    h, w = img_shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)

    if not os.path.exists(label_path):
        return mask  # imagem sem nenhuma instância anotada

    with open(label_path, "r") as f:
        lines = [l.strip() for l in f.readlines() if l.strip()]

    classes_aceitas = None
    if class_filter is not None:
        classes_aceitas = [class_filter] if isinstance(class_filter, int) else list(class_filter)

    for line in lines:
        values = line.split()
        class_id = int(float(values[0]))
        coords = list(map(float, values[1:]))

        if classes_aceitas is not None and class_id not in classes_aceitas:
            continue

        pts = np.array(
            [(coords[i] * w, coords[i + 1] * h) for i in range(0, len(coords), 2)],
            dtype=np.int32,
        )
        cv2.fillPoly(mask, [pts], 1)

    return mask


def predict_mask(model, image_path, conf=0.25, class_filter=None):
    """
    Roda a inferência do modelo treinado numa imagem e devolve uma máscara
    binária (0/1) já no tamanho original da imagem, combinando todas as
    instâncias detectadas (ou só as da(s) classe(s) em class_filter).
    """
    img = cv2.imread(image_path)
    h, w = img.shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)

    results = model.predict(image_path, conf=conf, verbose=False)
    r = results[0]

    if r.masks is None:
        return mask  # nenhuma instância detectada

    classes_aceitas = None
    if class_filter is not None:
        classes_aceitas = [class_filter] if isinstance(class_filter, int) else list(class_filter)

    for i, polygon in enumerate(r.masks.xy):  # já em coordenadas de pixel da imagem original
        if classes_aceitas is not None:
            cls_id = int(r.boxes.cls[i].item())
            if cls_id not in classes_aceitas:
                continue
        pts = polygon.astype(np.int32)
        cv2.fillPoly(mask, [pts], 1)

    return mask


def calculate_iou(mask_a, mask_b):
    """
    IoU (Intersection over Union) entre duas máscaras binárias do mesmo tamanho.
    Quando união == 0 (as duas máscaras vazias), retorna 1.0 em vez de NaN.
    """
    intersection = np.logical_and(mask_a, mask_b).sum()
    union = np.logical_or(mask_a, mask_b).sum()
    if union == 0:
        return 1.0
    return intersection / union

In [ ]:
import os, glob, yaml
import cv2
import numpy as np
import pandas as pd

if "DATASET_DIR" not in globals():
    DATASET_DIR = "/content/drive/MyDrive/LuisPedro/pistaClandestina/datasets/google_earth_v3_flip"  # <-- ajuste

if "data_cfg" not in globals():
    with open(os.path.join(DATASET_DIR, "data.yaml"), "r") as f:
        data_cfg = yaml.safe_load(f)

# IMPORTANTE: SPLIT é a CHAVE do data.yaml ('train', 'val' ou 'test'),
# não um caminho -- o caminho completo é montado logo abaixo a partir
# de DATASET_DIR + data_cfg[SPLIT]. Não coloque um path aqui.
SPLIT = "test"
if SPLIT not in data_cfg:
    SPLIT = "valid" if "valid" in data_cfg else list(data_cfg.keys())[0]

split_images_dir = os.path.join(DATASET_DIR, SPLIT, "images")
split_labels_dir = os.path.join(DATASET_DIR, SPLIT, "labels")

print(split_images_dir)
print(os.path.exists(split_images_dir))
print(len(glob.glob(os.path.join(split_images_dir, "*.jpg"))))

image_paths = sorted(glob.glob(os.path.join(split_images_dir, "*.*")))

resultados = []
for img_path in image_paths:
    filename = os.path.basename(img_path)
    label_path = os.path.join(split_labels_dir, os.path.splitext(filename)[0] + ".txt")

    img = cv2.imread(img_path)
    h, w = img.shape[:2]

    gt_mask = label_to_mask(label_path, (h, w))
    pred_mask = predict_mask(model, img_path, conf=0.25)

    iou = calculate_iou(gt_mask, pred_mask)
    resultados.append({"imagem": filename, "iou": iou})

df_iou = pd.DataFrame(resultados)
mean_iou = df_iou["iou"].mean()

print(df_iou)
print(f"\nIoU médio no split '{SPLIT}': {mean_iou:.4f}")

output_csv = os.path.join(DATASET_DIR, f"iou_{SPLIT}.csv")
df_iou.to_csv(output_csv, index=False)
print(f"Resultados salvos em: {output_csv}")